In [ ]:
#@title 按這裡開始（先按 ▶）
print('✅ W05 出發！本週目標：把 TM 訓練好的模型變成一個檔案，再用自己寫的程式批次跑它')
print('先確認左側檔案面板有 converted_keras.zip 與 test.zip，沒有就先上傳。')

# W05　訓練、匯出模型並批次推論

**先存副本**：`檔案 → 在雲端硬碟中儲存副本`，檔名 `AI導論_W05_學號_姓名`。

## 任務一：用電腦攝影機拍照與訓練（在瀏覽器做）

1. **開 TM 網站**：`teachablemachine.withgoogle.com` → Image
2. **允許攝影機權限**：Chrome 跳出視窗時按「允許」，按錯要到設定改
3. **類別改成英文名稱**：例如 `hat` 與 `nohat`，等一下資料夾要同名
4. **每類按住錄 50 張**：換角度、換距離、換光線，不要都同一張臉
5. **按 Train Model**：約 30 秒，訓練時不要切走或關掉分頁

**拍照的三個原則**：① 多樣性（換角度光線背景）② 兩類張數要平衡
③ **測試另外拍**——訓練用的畫面不能拿來測，另外用相機 App 拍 20 張。

## 任務二：匯出模型並上傳 Colab

1. **按 Export Model**（在 Preview 上方，訓練完成後才會亮）
2. **匯出格式選 Keras**：Tensorflow 分頁 → Keras，**不要選 .js**
3. **下載模型壓縮檔**：得到 `converted_keras.zip`，約 2-3 MB
4. **另外拍 20 張測試照**：用 Windows 相機 App，兩類各 10 張，壓成 `test.zip`
5. **兩個壓縮檔上傳**：拖進 Colab 左側檔案面板

> 測試照的資料夾結構要是 `test/hat/*.jpg`、`test/nohat/*.jpg`，
> **資料夾名稱必須和 TM 的類別名稱一模一樣**，任務四才算得出正確率。

### 先解壓縮兩個 zip

**會看到**：解完之後左側檔案面板會多出 `keras_model.h5`、`labels.txt` 與 `test/` 資料夾。

In [ ]:
!unzip -q -o converted_keras.zip
!unzip -q -o test.zip
!ls

## 任務二之二：把模型載進 Colab

TM 匯出的是舊版 Keras 格式，**前三行就是為了相容**。
裝完 `tf-keras` 如果報錯，先「執行階段 → 重新啟動工作階段」再從這一格重跑。

**會看到**：印出 `['hat', 'nohat']` 這樣的清單，就代表模型載好了。

In [ ]:
!pip install -q tf-keras
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # 相容 TM 舊格式

from tensorflow.keras.models import load_model
model = load_model('keras_model.h5', compile=False)

labels = [l.strip().split(' ', 1)[1]
          for l in open('labels.txt', encoding='utf-8')]
print(labels)      # 會印出你在 TM 取的兩個類別名稱

## 任務三：自己寫批次推論一整個資料夾

兩個 `____` 是本週重點：**前處理**與**取出分數最高的類別**。

- ① TM 的模型吃的是 −1 到 1 的數值，原始像素是 0 到 255，中間差一個換算
- ② `p` 是每一類的分數，要找出「分數最高的那一個是第幾類」

**寫對了會看到**：前五張的檔名與判斷結果一行一行印出來。

In [ ]:
from PIL import Image
import numpy as np, glob

def predict(path):
    img = Image.open(path).convert('RGB').resize((224, 224))
    x = np.asarray(img, dtype=np.float32)
    x = ____                       # ← ① 換算成 -1 到 1
    p = model.predict(x[None, ...], verbose=0)[0]
    return labels[____]            # ← ② 分數最高的那一類

files = sorted(glob.glob('test/*/*.jpg'))
print('共', len(files), '張')
for f in files[:5]:
    print(f, '→', predict(f))

## 任務四：自己寫正確率計算

三個 `____` 都要自己寫，**這一格不可以用 sklearn**。

**寫對了會看到**：`測 20 張，答對 18 張`、`正確率 = 0.90` 這樣兩行。
資料夾名稱要和 TM 的類別名稱完全一樣，否則永遠算 0 分。

In [ ]:
# 資料夾名稱就是正確答案：test/hat、test/nohat
right = 0

for f in files:
    truth = f.split('/')[1]
    pred = predict(f)
    if ____:                   # ← ① 判斷這一張猜對了沒
        right = ____           # ← ② 答對就加一

acc = ____                     # ← ③ 正確率 ＝ 答對 ÷ 總數
print(f'測 {len(files)} 張，答對 {right} 張')
print(f'正確率 = {acc:.2f}')

### 實驗記錄表（抄在紙上填）

| 實驗 | 每類張數 | 訓練 Epochs | 批次測出的正確率 |
|---|---|---|---|
| ① 少量資料 | 20 | 50 | |
| ② 標準 | 50 | 50 | |
| ③ 大量資料 | 100 | 50 | |
| ④ 換人來測 ② | 50 | 50 | |
| ⑤ 換背景測 ② | 50 | 50 | |
| 看出什麼（一句話） | | | |

---

## 任務五（進階）：自己寫混淆矩陣

**不可以用 sklearn**，用巢狀 `dict` 自己一格一格數。

**會看到**：對角線是答對的，對角線以外就看得出模型錯在哪一類。

In [ ]:
cm = {a: {b: 0 for b in labels} for a in labels}

for f in files:
    truth = f.split('/')[1]
    pred = predict(f)
    ____                       # ← ① 把這一格的計數加一

print('實際＼預測', labels)
for a in labels:
    print(a, [cm[a][b] for b in labels])

## 繳交：這一週要交什麼

模型檔、程式與數字三樣都要，缺一樣就重做不出來。

1. **`keras_model.h5` 與 `labels.txt` 存回雲端硬碟**：教室電腦會還原，留在本機下週就不見了
2. **筆記本要有任務三、四的完整輸出**：看得到前五張的判斷結果與最後的正確率
3. **實驗記錄表拍照上傳**：至少填完 ①②③ 三列，並寫出一句結論
4. **寫一句話：哪一種情況最容易判錯**：做完混淆矩陣的人，用矩陣的數字回答

### 常見狀況排除

| 狀況 | 原因 | 怎麼處理 |
|---|---|---|
| 攝影機是黑的 | 被視訊軟體佔用 | 關掉 Teams／Meet，再重新整理分頁 |
| 沒跳出權限視窗 | 之前按過「封鎖」 | 網址列左邊鎖頭圖示 → 相機 → 允許 |
| 載入模型就報錯 | Keras 版本不合 | 先跑 `tf-keras` 那三行，再重新啟動階段 |
| 找不到 h5 檔 | zip 沒解壓或路徑不對 | 先跑 `!ls` 看檔案在不在根目錄 |
| `files` 是空的 | 副檔名不是 `.jpg` | 把 glob 的 `*.jpg` 改成 `*.*` |
| `KeyError` 類別名 | 資料夾名與 TM 類別不同 | 資料夾改成和 `labels` 印出來的一樣 |
| 正確率高得離譜 | 拿訓練過的畫面來測 | 一定要另外拍一批新照片 |
| 下課後模型不見 | 檔案在執行階段暫存區 | 把 h5 與 zip 存回雲端硬碟 |

### 延伸挑戰（A 到 C 越後面越難）

- **A 改參數再跑**：回 TM 把每類補到 100 張重訓一次，用同一批測試照片再跑批次推論。正確率變多少？
- **B 換一種做法**：把 `predict()` 改成回傳兩個類別的機率，找出模型最猶豫的那三張照片。
- **C 說出為什麼**：換人測正確率就掉，為什麼？從混淆矩陣指出是哪一類被誤判，並說明要補拍什麼樣的照片。

---

<details>
<summary><b>參考解</b>（真的卡住再打開，先自己試滿 10 分鐘）</summary>

**任務三**

```python
    x = (x / 127.5) - 1            # ① 0~255 換算成 -1~1
    ...
    return labels[int(p.argmax())] # ② 分數最高的那一類
```

**任務四**

```python
    if pred == truth:              # ①
        right = right + 1          # ② 也可以寫 right += 1

acc = right / len(files)           # ③
```

**任務五**

```python
    cm[truth][pred] += 1           # ① 實際是 truth、被判成 pred 的那一格加一
```

</details>